# Getting Started with Promptolution

## Welcome to Promptolution! 

Discover a powerful tool for evolving and optimizing your LLM prompts. This notebook provides a friendly introduction to Promptolution's core functionality, by showcasing how you can easily find the best prompt to solve a classification problem.

We're excited to have you try Promptolution - let's get started!

## Installation
Install Promptolution with a single command

In [ ]:
! pip install promptolution

## Imports

In [ ]:
import pandas as pd
from promptolution.llms import APILLM
from promptolution.tasks import ClassificationTask
from promptolution.predictors import MarkerBasedPredictor
from promptolution.optimizers import CAPO
import nest_asyncio

nest_asyncio.apply()  # Required for notebook environments

## Setting Up Your Optimization

### Prepare the data

Below, we're using a subsample of the subjectivity dataset from Hugging Face as an example. When using your own dataset, simply ensure you name the input column "x" and the target column "y", and provide a brief description of your task, that will passed to the meta-llm during optimization.

In [ ]:
df = pd.read_csv("hf://datasets/tasksource/subjectivity/train.csv").sample(500)
df = df.rename(columns={"Sentence": "x", "Label": "y"})
df = df.replace({"OBJ": "objective", "SUBJ": "subjective"})

task_description = (
    "The dataset contains sentences labeled as either subjective or objective. "
    "The task is to classify each sentence as either subjective or objective. "
    "The class mentioned in between the answer tags <final_answer></final_answer> will be used as the prediction."
)

### Creating Inital Prompts

We've defined some starter prompts below, but you don't need to do this necessarily, since Promptolution can also automatically generates initial prompts based on your data or the provided task description.

In [ ]:
init_prompts = [
    'Classify the given text as either an objective or subjective statement based on the tone and language used: e.g. the tone and language used should indicate whether the statement is a neutral, factual summary (objective) or an expression of opinion or emotional tone (subjective). Include the output classes "objective" or "subjective" in the prompt.',
    "What kind of statement is the following text: [Insert text here]? Is it <objective_statement> or <subjective_statement>?",
    'Identify whether a sentence is objective or subjective by analyzing the tone, language, and underlying perspective. Consider the emotion, opinion, and bias present in the sentence. Are the authors presenting objective facts or expressing a personal point of view? The output will be either "objective" (output class: objective) or "subjective" (output class: subjective).',
    "Classify the following sentences as either objective or subjective, indicating the name of the output classes: [input sentence]. Output classes: objective, subjective",
    '_query a text about legal or corporate-related issues, and predict whether the tone is objective or subjective, outputting the corresponding class "objective" for non-subjective language or "subjective" for subjective language_',
    'Classify a statement as either "subjective" or "objective" based on whether it reflects a personal opinion or a verifiable fact. The output classes to include are "objective" and "subjective".',
    "Classify the text as objective or subjective based on its tone and language.",
    "Classify the text as objective or subjective based on the presence of opinions or facts. Output classes: objective, subjective.",
    "Classify the given text as objective or subjective based on its tone, focusing on its intention, purpose, and level of personal opinion or emotional appeal, with outputs including classes such as objective or subjective.",
    "Categorize the text as either objective or subjective, considering whether it presents neutral information or expresses a personal opinion/bias.\n\nObjective: The text has a neutral tone and presents factual information about the actions of Democrats in Congress and the union's negotiations.\n\nSubjective: The text has a evaluative tone and expresses a positive/negative opinion/evaluation about the past performance of the country.",
    'Given a sentence, classify it as either "objective" or "subjective" based on its tone and language, considering the presence of third-person pronouns, neutral language, and opinions. Classify the output as "objective" if the tone is neutral and detached, focusing on facts and data, or as "subjective" if the tone is evaluative, emotive, or biased.',
    'Identify whether the given sentence is subjective or objective, then correspondingly output "objective" or "subjective" in the form of "<output class>, (e.g. "objective"), without quotes. Please note that the subjective orientation typically describes a sentence where the writer expresses their own opinion or attitude, whereas an objective sentence presents facts or information without personal involvement or bias. <output classes: subjective, objective>',
]

### Configure Your LLM

Promptolution offers three flexible ways to access language models:

1. Local LLMs (using the Transformers library)
1. vLLM backend (for efficient serving of large language models)
1. API-based LLMs (compatible with any provider following the OpenAI standard)

For this demonstration, we'll use the DeepInfra API, but you can easily switch to other providers like Anthropic or OpenAI.

In [ ]:
api_key = "API_KEY"  # Replace with your API key

### Choose Your Components

Here's an explanation of the most important choices, and what we use in this example:
- `optimizer`: the algorithm used for prompt optimization. Here we use `CAPO`, as it is capable of leveraging few-shot examples.
- `llm`: the language model, used as both the *downstream* model (which makes the predictions) and the *meta* model (which proposes new prompts). Here an `APILLM` pointing at DeepInfra's `meta-llama/Meta-Llama-3-8B-Instruct`.
- `task`: wraps your `df` and defines the objective. `task_description` is a string describing the task, `n_subsamples` sets how many datapoints are used per evaluation step (here 30).
- `predictor`: how the label is extracted from the LLM output. Here from between markers, using the task's `classes`.
- `initial_prompts`: the prompts the optimizer starts from and improves. Here we pass `init_prompts`.
- `n_steps`: the number of optimization steps (here 10).

In [ ]:
llm = APILLM(
    api_url="https://api.deepinfra.com/v1/openai",
    model_id="meta-llama/Meta-Llama-3-8B-Instruct",
    api_key=api_key,
)
task = ClassificationTask(df, task_description=task_description, n_subsamples=30)
optimizer = CAPO(
    predictor=MarkerBasedPredictor(llm, classes=task.classes),
    meta_llm=llm,
    task=task,
    initial_prompts=init_prompts,
)

## Optimize Your Prompts

With everything built, you're ready to optimize! Calling `optimizer.optimize(n_steps=...)` runs the optimization loop and returns the best prompts. (To score prompts on a held-out split afterwards, see `promptolution.utils.evaluate_prompts`. Expect this cell to take a few minutes to run.

In [ ]:
prompts = optimizer.optimize(n_steps=10)

As you can see, most optimized prompts are semantically very similar, however they often differ heavily in performance. This is exactly what we observed in our experiments across various LLMs and datasets. Running prompt optimization is an easy way to gain significant performance improvements on your task for free!

If you run into any issues while using Promptolution, please feel free to contact us. We're also happy to receive support through pull requests and other contributions to the project.


Happy prompt optimizing! 🚀✨ We can't wait to see what you build with Promptolution! 🤖💡

In [ ]:
prompts